In [ ]:
# =========================
# 0. Install / Import
# =========================
!pip -q install pyspark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, CountVectorizer, IDF, StringIndexer
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# =========================
# 1. Create SparkSession
# =========================
spark = SparkSession.builder \
    .appName("DSOTM_Sentiment_Baseline") \
    .getOrCreate()

spark.conf.set("spark.sql.shuffle.partitions", "8")

# =========================
# 2. Load Data
# =========================
file_path = "../dsotm_reviews.csv"   # Change this path if needed
df = spark.read.csv(file_path, header=True, inferSchema=True)

print("Raw ")
df.printSchema()
print("Original sample count:", df.count())
df.show(5, truncate=100)

# =========================
# 3. Data Cleaning
#    - Keep Review / Rating
#    - Remove missing values
#    - Remove duplicates
#    - Remove empty text
# =========================
df = df.select("Review", "Rating")

df = df.dropna(subset=["Review", "Rating"])
df = df.dropDuplicates(["Review", "Rating"])

df = df.withColumn("Review", F.trim(F.col("Review")))
df = df.filter(F.col("Review") != "")

# Ensure Rating is double
df = df.withColumn("Rating", F.col("Rating").cast("double"))
df = df.dropna(subset=["Rating"])

print("Cleaned sample count:", df.count())

# =========================
# 4. Label Construction
#    Sentiment labels are generated from Rating
#    Note: the rule for 3.0 is configurable
# =========================
THREE_AS_POSITIVE = True

if THREE_AS_POSITIVE:
    df = df.withColumn(
        "Sentiment",
        F.when(F.col("Rating") >= 3.0, F.lit("positive")).otherwise(F.lit("negative"))
    )
else:
    df = df.withColumn(
        "Sentiment",
        F.when(F.col("Rating") > 3.0, F.lit("positive")).otherwise(F.lit("negative"))
    )

df.groupBy("Sentiment").count().show()

# =========================
# 5. Fixed Data Split
#    To ensure reproducibility, do not use randomSplit directly
#    Instead, create a stable split key based on content
# =========================
df = df.withColumn(
    "split_key",
    (F.abs(F.hash(F.col("Review"), F.col("Rating"))) % 100).cast("int")
)

train_df = df.filter(F.col("split_key") < 80)
test_df = df.filter(F.col("split_key") >= 80)

print("Training set size:", train_df.count())
print("Test set size:", test_df.count())

# =========================
# 6. Label Encoding
# =========================
label_indexer = StringIndexer(
    inputCol="Sentiment",
    outputCol="label",
    handleInvalid="skip"
)

# =========================
# 7. Text Preprocessing
#    - Lowercase
#    - Remove non-letter characters
#    - Tokenize
#    - Remove stopwords
# =========================
train_df = train_df.withColumn(
    "clean_text",
    F.lower(F.regexp_replace(F.col("Review"), r"[^a-zA-Z\s]", " "))
)

test_df = test_df.withColumn(
    "clean_text",
    F.lower(F.regexp_replace(F.col("Review"), r"[^a-zA-Z\s]", " "))
)

tokenizer = RegexTokenizer(
    inputCol="clean_text",
    outputCol="tokens",
    pattern="\\s+",
    gaps=True
)

remover = StopWordsRemover(
    inputCol="tokens",
    outputCol="filtered_tokens"
)

# =========================
# 8. TF-IDF Features
# =========================
vectorizer = CountVectorizer(
    inputCol="filtered_tokens",
    outputCol="raw_features",
    vocabSize=10000,
    minDF=2.0
)

idf = IDF(
    inputCol="raw_features",
    outputCol="features"
)

# =========================
# 9. Logistic Regression Baseline
# =========================
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=100,
    regParam=0.0,
    elasticNetParam=0.0
)

pipeline = Pipeline(stages=[
    label_indexer,
    tokenizer,
    remover,
    vectorizer,
    idf,
    lr
])

# =========================
# 10. Train the Model
# =========================
model = pipeline.fit(train_df)

# =========================
# 11. Prediction
# =========================
predictions = model.transform(test_df)

predictions.select(
    "Review", "Rating", "Sentiment", "label", "prediction", "probability"
).show(10, truncate=100)

# =========================
# 12. Evaluation
# =========================
accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

precision_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

recall_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
)

accuracy = accuracy_evaluator.evaluate(predictions)
f1 = f1_evaluator.evaluate(predictions)
precision = precision_evaluator.evaluate(predictions)
recall = recall_evaluator.evaluate(predictions)

print("Test Accuracy:", round(accuracy, 4))
print("Test Precision:", round(precision, 4))
print("Test Recall:", round(recall, 4))
print("Test F1:", round(f1, 4))


# =========================
# 13. Check Label Mapping
# =========================
label_model = model.stages[0]
print("Label order (StringIndexer):", label_model.labels)

# =========================
# 14. Save Fixed Splits and Prediction Results
# =========================
train_df.select("Review", "Rating", "Sentiment") \
    .coalesce(1) \
    .write.mode("overwrite") \
    .option("header", True) \
    .csv("/content/train_fixed_split")

test_df.select("Review", "Rating", "Sentiment") \
    .coalesce(1) \
    .write.mode("overwrite") \
    .option("header", True) \
    .csv("/content/test_fixed_split")

predictions.select(
    "Review", "Rating", "Sentiment", "label", "prediction"
).coalesce(1) \
 .write.mode("overwrite") \
 .option("header", True) \
 .csv("/content/test_predictions")

print("Saved outputs:")
print("/content/train_fixed_split")
print("/content/test_fixed_split")
print("/content/test_predictions")


原始数据：
root
 |-- Review: string (nullable = true)
 |-- Rating: double (nullable = true)

原始样本数： 1548
+----------------------------------------------------------------------------------------------------+------+
|                                                                                              Review|Rating|
+----------------------------------------------------------------------------------------------------+------+
|"""More has been said about Dark Side of the Moon than will ever be necessary both positive and n...|   4.5|
|What can I possibly say about an album that not only means so much to so many and has influenced ...|   5.0|
|"You know for a band that spent several albums trying to atone for unceremoniously ousting its fo...|   2.0|
|"          Has finally clicked with me in full after 30 years absolutely smashing the previous re...|   4.0|
|"          So why are people afraid to say this isnt a masterpiece but is basically a well-crafte...|   4.5|
+-------------------